In [2]:
import pandas as pd
import numpy as np

# Load data
expr = pd.read_csv("results/normalized_expression.csv")
meta = pd.read_csv("results/sample_metadata.csv")

# Extract subject + timepoint
meta["timepoint"] = meta["characteristics"].str.extract(r"t(\d+)").astype(int)
meta["subject"] = meta["title"].str.extract(r"subject (\d+)")
meta["condition"] = meta["condition"].astype(str)

def assign_phase(t):
    if 1 <= t <= 4:
        return "baseline"
    elif 5 <= t <= 8:
        return "experimental"
    elif 9 <= t <= 12:
        return "recovery"
    return np.nan

meta["phase"] = meta["timepoint"].apply(assign_phase)

print(expr.shape)
print(meta[["sample_id", "subject", "timepoint", "phase", "condition"]].head())

(8506, 164)
    sample_id subject  timepoint         phase     condition
0  GSM2600155    6045          1      baseline  normal_sleep
1  GSM2600156    6045          2      baseline  normal_sleep
2  GSM2600157    6045          3      baseline  normal_sleep
3  GSM2600158    6045          4      baseline  normal_sleep
4  GSM2600159    6045          5  experimental  normal_sleep


In [4]:
# Long format
expr_long = expr.melt(id_vars=["ID_REF"], var_name="sample_id", value_name="expression")

full_df = expr_long.merge(
    meta[["sample_id", "subject", "condition", "timepoint", "phase"]],
    on="sample_id",
    how="left"
)

baseline_ref = (
    full_df[full_df["phase"] == "baseline"]
    .groupby(["ID_REF", "subject", "condition"])["expression"]
    .mean()
    .reset_index()
    .rename(columns={"expression": "baseline_expr"})
)

norm_df = full_df.merge(
    baseline_ref,
    on=["ID_REF", "subject", "condition"],
    how="left"
)

# Ratio normalization to baseline
norm_df["norm_expr"] = norm_df["expression"] / (norm_df["baseline_expr"] + 1e-8)

print(norm_df.head())
print(norm_df.shape)

    ID_REF   sample_id  expression subject     condition  timepoint     phase  \
0  7896742  GSM2600155    7.609771    6045  normal_sleep          1  baseline   
1  7896746  GSM2600155    8.011875    6045  normal_sleep          1  baseline   
2  7896748  GSM2600155    4.970357    6045  normal_sleep          1  baseline   
3  7896750  GSM2600155    4.681245    6045  normal_sleep          1  baseline   
4  7896752  GSM2600155    8.864139    6045  normal_sleep          1  baseline   

   baseline_expr  norm_expr  
0       7.370752   1.032428  
1       8.834650   0.906870  
2       5.988404   0.829997  
3       5.096625   0.918499  
4       9.189856   0.964557  
(1386478, 9)


In [6]:
# Keep only experimental phase
exp_df = norm_df[norm_df["phase"] == "experimental"].copy()

subject_gene = (
    exp_df.groupby(["subject", "condition", "ID_REF"])["norm_expr"]
    .mean()
    .reset_index()
)

ml_df = subject_gene.pivot_table(
    index=["subject", "condition"],
    columns="ID_REF",
    values="norm_expr"
).reset_index()

print(ml_df.shape)
print(ml_df.iloc[:5, :8])

(14, 8508)
ID_REF subject       condition   7896742   7896746   7896748   7896750  \
0         6045    normal_sleep  0.990723  1.016201  1.057685  1.037041   
1         6068    normal_sleep  1.063656  1.023242  1.073950  1.088754   
2         6089    normal_sleep  1.010974  1.016215  1.061090  1.030508   
3         6166  sleep_deprived  1.009613  0.978909  0.998676  0.974846   
4         6207  sleep_deprived  0.951161  1.062021  1.184585  1.158109   

ID_REF   7896752   7896817  
0       1.021647  0.929888  
1       1.023345  0.957097  
2       0.993871  1.020070  
3       1.005649  0.993019  
4       1.024869  1.061306  


In [8]:
# Read platform annotation
annot = pd.read_csv(
    "results/GPL6244_family.soft",
    sep="\t",
    skiprows=40608,
    nrows=33297,
    low_memory=False
)

annot["ID_REF"] = annot["ID"].astype(str)
annot["Gene Symbol"] = annot["gene_assignment"].str.split(" // ").str[1]
annot["Gene Symbol"] = annot["Gene Symbol"].replace("---", pd.NA)
annot = annot[["ID_REF", "Gene Symbol"]].dropna()

probe_to_symbol = dict(zip(annot["ID_REF"], annot["Gene Symbol"]))

feature_cols = [c for c in ml_df.columns if c not in ["subject", "condition"]]

rename_map = {c: probe_to_symbol.get(str(c), str(c)) for c in feature_cols}
ml_df = ml_df.rename(columns=rename_map)

print(ml_df.iloc[:3, :8])

ID_REF subject     condition  LOC728323     MT-TM     MT-TW     MT-TD  \
0         6045  normal_sleep   0.990723  1.016201  1.057685  1.037041   
1         6068  normal_sleep   1.063656  1.023242  1.073950  1.088754   
2         6089  normal_sleep   1.010974  1.016215  1.061090  1.030508   

ID_REF     MT-TK     ISG15  
0       1.021647  0.929888  
1       1.023345  0.957097  
2       0.993871  1.020070  


In [10]:
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Target
ml_df["y"] = (ml_df["condition"] == "sleep_deprived").astype(int)

X = ml_df.drop(columns=["subject", "condition", "y"])
y = ml_df["y"]

print("Subjects:", len(ml_df))
print("Cases:", y.sum(), "Controls:", (1 - y).sum())
print("Number of features:", X.shape[1])

Subjects: 14
Cases: 8 Controls: 6
Number of features: 8506


In [12]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

log_reg_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("select", SelectKBest(score_func=f_classif, k=min(20, X.shape[1]))),
    ("clf", LogisticRegression(
        penalty="l1",
        solver="liblinear",
        max_iter=5000,
        random_state=42
    ))
])

rf_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("select", SelectKBest(score_func=f_classif, k=min(20, X.shape[1]))),
    ("clf", RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_leaf=1,
        random_state=42
    ))
])

scoring = {
    "accuracy": "accuracy",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

log_reg_scores = cross_validate(
    log_reg_pipe, X, y, cv=cv, scoring=scoring, return_train_score=False
)

rf_scores = cross_validate(
    rf_pipe, X, y, cv=cv, scoring=scoring, return_train_score=False
)

def summarize_scores(name, scores):
    print(f"\n{name}")
    for metric in ["test_accuracy", "test_f1", "test_roc_auc"]:
        vals = scores[metric]
        print(f"{metric}: {vals.mean():.3f} ± {vals.std():.3f}")

summarize_scores("L1 Logistic Regression", log_reg_scores)
summarize_scores("Random Forest", rf_scores)


L1 Logistic Regression
test_accuracy: 0.500 ± 0.082
test_f1: 0.357 ± 0.254
test_roc_auc: 0.444 ± 0.079

Random Forest
test_accuracy: 0.367 ± 0.125
test_f1: 0.467 ± 0.144
test_roc_auc: 0.417 ± 0.245


In [14]:
log_reg_pipe.fit(X, y)

# Get selected features
selected_mask = log_reg_pipe.named_steps["select"].get_support()
selected_features = X.columns[selected_mask]

# Get coefficients
coefs = log_reg_pipe.named_steps["clf"].coef_.ravel()

coef_df = pd.DataFrame({
    "Gene": selected_features,
    "Coefficient": coefs,
    "AbsCoefficient": np.abs(coefs)
}).sort_values("AbsCoefficient", ascending=False)

print(coef_df.head(20))

       Gene  Coefficient  AbsCoefficient
8   SNORD57     0.549451        0.549451
9      CST7    -0.520690        0.520690
1     ARL5B     0.501941        0.501941
15     CCM2     0.425878        0.425878
5   TMEM104    -0.245165        0.245165
12    VDAC1    -0.204277        0.204277
17  TMEM209    -0.200236        0.200236
0     STX12    -0.180291        0.180291
11     EGR1    -0.169286        0.169286
6   PSTPIP2    -0.075292        0.075292
14    PRDM1     0.000000        0.000000
18  C9orf16     0.000000        0.000000
16     GUSB     0.000000        0.000000
10  BHLHE40     0.000000        0.000000
13     DND1     0.000000        0.000000
7     HADHB     0.000000        0.000000
4      RCN2     0.000000        0.000000
3      CCNK     0.000000        0.000000
2     RECQL     0.000000        0.000000
19     GLE1     0.000000        0.000000


In [16]:
rf_pipe.fit(X, y)

selected_mask_rf = rf_pipe.named_steps["select"].get_support()
selected_features_rf = X.columns[selected_mask_rf]

importances = rf_pipe.named_steps["clf"].feature_importances_

rf_imp_df = pd.DataFrame({
    "Gene": selected_features_rf,
    "Importance": importances
}).sort_values("Importance", ascending=False)

print(rf_imp_df.head(20))

       Gene  Importance
7     HADHB    0.132172
12    VDAC1    0.110993
5   TMEM104    0.076567
13     DND1    0.070964
4      RCN2    0.064069
9      CST7    0.060069
17  TMEM209    0.059901
15     CCM2    0.056567
3      CCNK    0.049267
8   SNORD57    0.043498
14    PRDM1    0.043333
0     STX12    0.033402
10  BHLHE40    0.032527
19     GLE1    0.031669
16     GUSB    0.031370
2     RECQL    0.031358
1     ARL5B    0.028433
6   PSTPIP2    0.021370
11     EGR1    0.015802
18  C9orf16    0.006667


In [18]:
coef_df.to_csv("results/ml_logistic_top_genes.csv", index=False)
rf_imp_df.to_csv("results/ml_random_forest_top_genes.csv", index=False)

print("Saved:")
print("- results/ml_logistic_top_genes.csv")
print("- results/ml_random_forest_top_genes.csv")

Saved:
- results/ml_logistic_top_genes.csv
- results/ml_random_forest_top_genes.csv


In [20]:
candidate_genes = [
    "RCN2", "IL11RA", "TAF11", "RSAD1", "PPP1R14B",
    "ZNF655", "MAPK14", "GUSB", "STX12", "CPEB4"
]

candidate_genes = [g for g in candidate_genes if g in X.columns]

X_small = X[candidate_genes].copy()

print("Candidate genes used:", X_small.columns.tolist())
print("Shape:", X_small.shape)

Candidate genes used: ['RCN2', 'IL11RA', 'TAF11', 'RSAD1', 'PPP1R14B', 'ZNF655', 'MAPK14', 'GUSB', 'STX12', 'CPEB4']
Shape: (14, 10)
